# This file will find the closest (straight-line distance) station for each properties as the extra features, store station location & distance

In [1]:
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
from sklearn.neighbors import BallTree
from geopy.distance import great_circle

In [2]:
# read the files record propertry and station respectively
property_data_origin = pd.read_csv("../data/curated/merged_data/merged_data_with_facility.csv")
station_data = pd.read_csv("../data/raw/external_data/PTV_station_data.csv") # change path later
property_data =  property_data_origin[['name','coordinates']]
property_data

,name,coordinates
0,31 Chittagong Drive Clyde North VIC 3978,"[-38.1053122, 145.3570863]"
1,50 Elmtree Crescent Clyde North VIC 3978,"[-38.0825712, 145.3561984]"
2,7 Mortdale Lane Clyde North VIC 3978,"[-38.0961758, 145.3800644]"
3,54 Walhallow Drive Clyde North VIC 3978,"[-38.1133324, 145.3457396]"
4,10 Sicily Road Clyde North VIC 3978,"[-38.1295789, 145.3642993]"
...,...,...
8797,61 Tongue Street Yarraville VIC 3013,"[-37.8131463, 144.8909053]"
8798,47 Mill Avenue Yarraville VIC 3013,"[-37.8222822, 144.872198]"
8799,12 Adeney Street Yarraville VIC 3013,"[-37.8163817, 144.8666543]"
8800,229B Somerville Road Yarraville VIC 3013,"[-37.8124289, 144.8779569]"


In [3]:
station_data

,station_name,station_coordinates
0,Dimboola,"[-36.452019916533985, 142.03097879572368]"
1,Kerang,"[-35.73311015757247, 143.92465096653459]"
2,Echuca,"[-36.13103835849756, 144.75328509216718]"
3,Rochester,"[-36.36220456920009, 144.69855282297706]"
4,Elmore,"[-36.49489611863695, 144.60763619157052]"
...,...,...
326,Gardiner,"[-37.852867997222994, 145.0508099029106]"
327,Jordanville,"[-37.873708994762815, 145.11246954380377]"
328,Mount Waverley,"[-37.87532263838918, 145.1281391923223]"
329,South Geelong,"[-38.15893436083373, 144.35965242241852]"


In [4]:
# This function will accept a string to parse coordinate string into point object
def parse_coordinates(coord_str):
    parts = coord_str.strip('[]').split(',')
    return (float(parts[0]), float(parts[1]))

# parse coordinate string into point object for both property & station data
closest_station = []
station_distance = []
property_coordinates = property_data['coordinates'].apply(parse_coordinates)
all_station_coordinates = station_data['station_coordinates'].apply(parse_coordinates)

# find the closest station for each property
for property_coord in property_coordinates:
    # By default: no nearest station and the distance is positive infinity.
    min_distance = float('inf')
    closest_station_geo = None
    
    for i, station_coord in enumerate(all_station_coordinates):
        # check if the value of the coordinate point is valid
        if np.isnan(property_coord).any() or np.isnan(station_coord).any():
            continue
        # claculate the distance between property and station, update if it is smallest
        distance = great_circle(property_coord, station_coord).kilometers
        if distance < min_distance:
            min_distance = distance
            closest_station_geo = station_data.loc[i, 'station_coordinates']
    
    # add feature to store closest station for the property
    closest_station.append(closest_station_geo)
    station_distance.append(min_distance)
property_data['closest_station'] = closest_station
property_data['station_distance(KM)'] = station_distance
property_data

/tmp/ipykernel_125172/4177316917.py:31: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  property_data['closest_station'] = closest_station
/tmp/ipykernel_125172/4177316917.py:32: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  property_data['station_distance(KM)'] = station_distance


,name,coordinates,closest_station,station_distance(KM)
0,31 Chittagong Drive Clyde North VIC 3978,"[-38.1053122, 145.3570863]","[-38.05104856543076, 145.3665303282505]",6.090210
1,50 Elmtree Crescent Clyde North VIC 3978,"[-38.0825712, 145.3561984]","[-38.05104856543076, 145.3665303282505]",3.619981
2,7 Mortdale Lane Clyde North VIC 3978,"[-38.0961758, 145.3800644]","[-38.0663192834778, 145.4116572812351]",4.320648
3,54 Walhallow Drive Clyde North VIC 3978,"[-38.1133324, 145.3457396]","[-38.0995872148084, 145.2804738077291]",5.911467
4,10 Sicily Road Clyde North VIC 3978,"[-38.1295789, 145.3642993]","[-38.0995872148084, 145.2804738077291]",8.056215
...,...,...,...,...
8797,61 Tongue Street Yarraville VIC 3013,"[-37.8131463, 144.8909053]","[-37.81567190405462, 144.89006145414797]",0.290453
8798,47 Mill Avenue Yarraville VIC 3013,"[-37.8222822, 144.872198]","[-37.83072411396968, 144.88584876958876]",1.522703
8799,12 Adeney Street Yarraville VIC 3013,"[-37.8163817, 144.8666543]","[-37.7993395010312, 144.86344949623717]",1.915810
8800,229B Somerville Road Yarraville VIC 3013,"[-37.8124289, 144.8779569]","[-37.81567190405462, 144.89006145414797]",1.122803


In [5]:
# store closest station information to the merged data
property_data_origin['closest_station'] = closest_station
property_data_origin['station_distance(KM)'] = station_distance
# save as a CSV file
property_data_origin.to_csv("../data/curated/merged_data/merged_data_with_facility.csv", index=False)
merged_data_with_facility = pd.read_csv("../data/curated/merged_data/merged_data_with_facility.csv")
merged_data_with_facility

,name,rental_price,num_bedroom,num_bathroom,num_parking,postcode,coordinates,property_geometry,sa2_code,sa2_name,...,closest_school,school_distance(KM),closest_hospital,hospital_distance(KM),closest_mall,mall_distance(KM),closest_park,park_distance(KM),closest_station,station_distance(KM)
0,31 Chittagong Drive Clyde North VIC 3978,575.0,4,2,2.0,3978.0,"[-38.1053122, 145.3570863]",POINT (145.3570863 -38.1053122),212031556.0,Clyde North - South,...,"[-38.10602, 145.37876]",1.898006,"[-38.045325, 145.347181]",6.726397,"[-38.1184718, 145.3213262]",3.453903,"[-38.13958090963712, 145.36252669303045]",3.840116,"[-38.05104856543076, 145.3665303282505]",6.090210
1,50 Elmtree Crescent Clyde North VIC 3978,560.0,4,2,2.0,3978.0,"[-38.0825712, 145.3561984]",POINT (145.3561984 -38.0825712),212031555.0,Clyde North - North,...,"[-38.08468, 145.3638]",0.705427,"[-38.045325, 145.347181]",4.216162,"[-38.0604189, 145.3394612]",2.866025,"[-38.03401014838179, 145.37166338662394]",5.566924,"[-38.05104856543076, 145.3665303282505]",3.619981
2,7 Mortdale Lane Clyde North VIC 3978,490.0,2,2,1.0,3978.0,"[-38.0961758, 145.3800644]",POINT (145.3800644 -38.0961758),212031556.0,Clyde North - South,...,"[-38.10602, 145.37876]",1.100561,"[-38.045325, 145.347181]",6.344909,"[-38.0604189, 145.3394612]",5.332842,"[-38.13958090963712, 145.36252669303045]",5.064419,"[-38.0663192834778, 145.4116572812351]",4.320648
3,54 Walhallow Drive Clyde North VIC 3978,540.0,4,2,1.0,3978.0,"[-38.1133324, 145.3457396]",POINT (145.3457396 -38.1133324),212031556.0,Clyde North - South,...,"[-38.11488, 145.33828]",0.674921,"[-38.113312, 145.280832]",5.678594,"[-38.1184718, 145.3213262]",2.210922,"[-38.13958090963712, 145.36252669303045]",3.267265,"[-38.0995872148084, 145.2804738077291]",5.911467
4,10 Sicily Road Clyde North VIC 3978,520.0,4,2,2.0,3978.0,"[-38.1295789, 145.3642993]",POINT (145.3642993 -38.1295789),212031303.0,Cranbourne South,...,"[-38.12955, 145.33886]",2.225124,"[-38.113312, 145.280832]",7.522231,"[-38.1184718, 145.3213262]",3.956745,"[-38.13958090963712, 145.36252669303045]",1.122928,"[-38.0995872148084, 145.2804738077291]",8.056215
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8797,61 Tongue Street Yarraville VIC 3013,630.0,2,1,0.0,3013.0,"[-37.8131463, 144.8909053]",POINT (144.8909053 -37.8131463),213031352.0,Yarraville,...,"[-37.8137, 144.8899]",0.107655,"[-37.800508, 144.895155]",1.454065,"[-37.8016476, 144.8977057]",1.411290,"[-37.83084513979606, 144.91345002848863]",2.791844,"[-37.81567190405462, 144.89006145414797]",0.290453
8798,47 Mill Avenue Yarraville VIC 3013,730.0,4,3,2.0,3013.0,"[-37.8222822, 144.872198]",POINT (144.872198 -37.8222822),213031352.0,Yarraville,...,"[-37.82104, 144.87443]",0.239821,"[-37.797407, 144.887421]",3.072331,"[-37.828989, 144.84627]",2.396280,"[-37.83084513979606, 144.91345002848863]",3.746179,"[-37.83072411396968, 144.88584876958876]",1.522703
8799,12 Adeney Street Yarraville VIC 3013,450.0,3,1,2.0,3013.0,"[-37.8163817, 144.8666543]",POINT (144.8666543 -37.8163817),213031352.0,Yarraville,...,"[-37.8126, 144.87466]",0.819385,"[-37.797407, 144.887421]",2.789294,"[-37.828989, 144.84627]",2.273966,"[-37.83084513979606, 144.91345002848863]",4.413664,"[-37.7993395010312, 144.86344949623717]",1.915810
8800,229B Somerville Road Yarraville VIC 3013,300.0,1,1,0.0,3013.0,"[-37.8124289, 144.8779569]",POINT (144.8779569 -37.8124289),213031352.0,Yarraville,...,"[-37.8126, 144.87466]",0.290245,"[-37.797407, 144.887421]",1.865866,"[-37.8016476, 144.8977057]",2.108881,"[-37.83084513979606, 144.91345002848863]",3.729966,"[-37.81567190405462, 144.89006145414797]",1.122803


In [7]:
# check if nan exist for distance features
nan_check = merged_data_with_facility.isna().any()
nan_columns = merged_data_with_facility.columns[merged_data_with_facility.isna().any()].tolist()
print("Columns with NaN values:", nan_columns)
nan_counts = merged_data_with_facility.isna().sum()
print("NaN value counts per column:", nan_counts)

Columns with NaN values: ['sa2_code', 'sa2_name', 'sa2_geometry', '2021_population', 'avg_pop_growth_rates(%)']
NaN value counts per column: name                         0
rental_price                 0
num_bedroom                  0
num_bathroom                 0
num_parking                  0
postcode                     0
coordinates                  0
property_geometry            0
sa2_code                     8
sa2_name                     8
sa2_geometry                 8
personal_income              0
avg_income_growth_rate(%)    0
2021_population              8
avg_pop_growth_rates(%)      9
crime_rate(%)                0
closest_school               0
school_distance(KM)          0
closest_hospital             0
hospital_distance(KM)        0
closest_mall                 0
mall_distance(KM)            0
closest_park                 0
park_distance(KM)            0
closest_station              0
station_distance(KM)         0
dtype: int64
